# 04 · Channel economics

Customer value by acquisition channel on a **26-week window**, using only customers with ≥ 26 weeks of observation so immature 2025 cohorts are not compared with mature 2023 ones. A sensitivity check re-runs retention at 12 and 52 weeks.

Supports deck slides **5, 9, A1, A8**.

![Channel value vs retention](../reports/figures/s5_channel_bubble.png)

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from common import load, customers, weekly_new, CHANNEL_ORDER, LAST_WEEK_IDX
pd.set_option("display.width", 140)
df = load()
c = customers(df)

In [2]:
first = df[df.is_first_delivery == 1].set_index("customer_id")
c["first_disc_pct"] = first.discount_amount / first.gross_revenue

W = 26
mature = c[c.obs_weeks >= W]
rev26 = df[df.tenure < W].groupby("customer_id").net_revenue.sum()
mature = mature.assign(rev26=rev26.reindex(mature.index).fillna(0),
                       ret26=(mature.last_tenure >= W),
                       repeat=(mature.n_deliv >= 2))
tab = mature.groupby("channel").agg(customers_26wk=("rev26", "size"), net_rev_per_cust_26wk=("rev26", "mean"),
                                    retention_26wk=("ret26", "mean"), first_order_discount=("first_disc_pct", "mean"),
                                    repeat_rate=("repeat", "mean")).reindex(CHANNEL_ORDER)
tab.insert(0, "customers_acquired", c.channel.value_counts().reindex(CHANNEL_ORDER))
for col in ("retention_26wk", "first_order_discount", "repeat_rate"):
    tab[col] = (tab[col] * 100).round(1)
tab["net_rev_per_cust_26wk"] = tab["net_rev_per_cust_26wk"].round(0)
print(f"Customers with >= 26 weeks of observation: {len(mature):,} of {len(c):,}")
tab

Customers with >= 26 weeks of observation: 7,447 of 9,062


,customers_acquired,customers_26wk,net_rev_per_cust_26wk,retention_26wk,first_order_discount,repeat_rate
channel,,,,,,
Organic,524,389,553.0,10.0,28.7,77.4
Paid Social,2766,2333,459.0,8.2,28.5,71.1
TV,1646,1363,447.0,5.8,28.7,67.9
Paid Search,1553,1319,395.0,6.1,29.0,68.5
Affiliate,1110,879,348.0,4.2,28.9,66.4
Referral,1463,1164,285.0,2.2,56.7,61.3


`customers_acquired` counts every customer; all value metrics use the ≥ 26-week subset (`customers_26wk`).

## Retention sensitivity by window

In [3]:
for win in (12, 26, 52):
    m = c[c.obs_weeks >= win]
    r = (m.last_tenure >= win).groupby(m.channel).mean().mul(100).round(1)
    print(f"{win}-week retention:", r.reindex(CHANNEL_ORDER).to_dict())
    print("   ranking:", " > ".join(r.sort_values(ascending=False).index))

12-week retention: {'Organic': 19.0, 'Paid Social': 14.4, 'TV': 10.9, 'Paid Search': 11.4, 'Affiliate': 8.6, 'Referral': 5.4}
   ranking: Organic > Paid Social > Paid Search > TV > Affiliate > Referral
26-week retention: {'Organic': 10.0, 'Paid Social': 8.2, 'TV': 5.8, 'Paid Search': 6.1, 'Affiliate': 4.2, 'Referral': 2.2}
   ranking: Organic > Paid Social > Paid Search > TV > Affiliate > Referral
52-week retention: {'Organic': 5.4, 'Paid Social': 3.1, 'TV': 2.2, 'Paid Search': 2.9, 'Affiliate': 0.9, 'Referral': 0.8}
   ranking: Organic > Paid Social > Paid Search > TV > Affiliate > Referral
